# 02 — Matplotlib for Signals and Systems

Every result in this course is a picture: a waveform, a spectrum, a Bode plot, a pole-zero
map. Matplotlib makes all of them, but it has two APIs and most confusion comes from mixing
them.

**Use the object-oriented API.** `fig, ax = plt.subplots()` then `ax.plot(...)`. The
`plt.plot(...)` state-machine style is fine for a throwaway one-liner and a liability the
moment you have more than one panel.

---

## Contents

| § | Topic |
|---|-------|
| 1 | Figure and Axes: the only structure you need to learn |
| 2 | Plotting waveforms properly |
| 3 | `stem` for discrete-time signals |
| 4 | Multi-panel layouts and shared axes |
| 5 | Log scales and decibels |
| 6 | The Bode plot |
| 7 | Pole-zero maps (s-plane and z-plane) |
| 8 | Spectrograms and 2-D data |
| 9 | Annotation: making the plot say something |
| 10 | Styling, rcParams, and colour |
| 11 | Saving figures for reports |
| 12 | Animation |
| 13 | Exercises |

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker

print("matplotlib", plt.matplotlib.__version__)

def time_vector(fs, duration):
    n = int(round(fs * duration))
    return np.arange(n) / fs, n

---
## 1. Figure and Axes: the only structure you need to learn

- **Figure** — the whole canvas. Owns the size, the DPI, the file you save.
- **Axes** — one plot region with its own x/y axis, title, legend. A figure holds one or many.

Almost every method you want is on the *Axes*: `ax.plot`, `ax.set_xlabel`, `ax.legend`,
`ax.set_xlim`. Learn that and the library stops feeling arbitrary.

In [ ]:
t, N = time_vector(500, 1.0)
x = np.sin(2 * np.pi * 3 * t)

fig, ax = plt.subplots(figsize=(9, 3))   # one figure, one axes

ax.plot(t, x)
ax.set_xlabel("time [s]")
ax.set_ylabel("amplitude")
ax.set_title("The anatomy of a plot")
ax.grid(alpha=0.3)

print("figure :", type(fig).__name__)
print("axes   :", type(ax).__name__)
print("lines on this axes:", ax.lines)

> **`plt.subplots()` returns a tuple.** With no arguments you get one Axes. With
> `plt.subplots(2, 3)` you get a 2×3 NumPy array of Axes, indexed `ax[row, col]`. With
> `plt.subplots(3)` you get a 1-D array indexed `ax[i]`. Watch for the case where one
> dimension is 1 — `plt.subplots(1, 2)` gives a 1-D array, not 2-D.

---
## 2. Plotting waveforms properly

A waveform plot that a marker can read has: axis labels **with units**, a title stating what
the signal is, a legend if there is more than one trace, and sensible limits.

In [ ]:
fs = 1000.0
t, N = time_vector(fs, 0.05)

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(t * 1000, np.sin(2 * np.pi * 50 * t), label="50 Hz", lw=1.6)
ax.plot(t * 1000, 0.6 * np.sin(2 * np.pi * 120 * t), label="120 Hz", lw=1.6, ls="--")
ax.plot(t * 1000, np.sin(2 * np.pi * 50 * t) + 0.6 * np.sin(2 * np.pi * 120 * t),
        label="sum", lw=1.0, color="0.3")

ax.set_xlabel("time [ms]")            # units in the label, always
ax.set_ylabel("amplitude [V]")
ax.set_title("Superposition of two sinusoids")
ax.legend(loc="upper right", ncol=3, fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 50)
ax.axhline(0, color="k", lw=0.6)      # a zero line helps the eye
fig.tight_layout()

### Line styling cheat-sheet

| argument | effect |
|----------|--------|
| `lw=1.5` | line width |
| `ls='--'` | style: `'-'`, `'--'`, `'-.'`, `':'` |
| `color='tab:red'` or `'0.4'` | named / grey level |
| `alpha=0.5` | transparency — invaluable for dense signals |
| `marker='o', ms=3` | sample markers |
| `zorder=0` | draw behind everything else |
| `label='...'` | picked up by `ax.legend()` |

In [ ]:
# Transparency is the fix for a plot that is too dense to read.
rng = np.random.default_rng(0)
t, N = time_vector(2000, 1.0)
clean = np.sin(2 * np.pi * 4 * t)

fig, ax = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for a, alpha in zip(ax, [1.0, 0.25]):
    for _ in range(30):
        a.plot(t, clean + rng.normal(0, 0.3, N), color="tab:blue", lw=0.5, alpha=alpha)
    a.plot(t, clean, "k", lw=2)
    a.set_title(f"30 noisy trials, alpha = {alpha}")
    a.set_xlabel("time [s]")
fig.tight_layout()

---
## 3. `stem` for discrete-time signals

When the signal is a sequence $x[n]$ rather than a sampled continuous signal, `stem` is the
honest representation. It shows that values exist only at integer indices.

In [ ]:
n = np.arange(-4, 21)
h = (0.8 ** n) * (n >= 0)

fig, ax = plt.subplots(figsize=(9, 3))
markerline, stemlines, baseline = ax.stem(n, h, basefmt=" ")
plt.setp(markerline, markersize=5, color="tab:blue")
plt.setp(stemlines, linewidth=1.2, color="tab:blue", alpha=0.7)

ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("n [samples]")
ax.set_ylabel("h[n]")
ax.set_title("Impulse response h[n] = 0.8ⁿ·u[n]")
ax.set_xticks(np.arange(-4, 21, 2))
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()

> **`stem` returns three objects** — the markers, the vertical lines, the baseline. Capture
> them and use `plt.setp` to style them; there is no single `color=` argument that reaches
> all three consistently across versions. `basefmt=" "` hides the baseline so you can draw
> your own with `axhline`.

In [ ]:
# Discrete convolution, shown as a three-panel story: input, impulse response, output.
x = np.array([1., 2., 3., 2., 1.])
h = np.array([1., -1.])
y = np.convolve(x, h)

fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
for a, (sig, name) in zip(ax, [(x, "x[n]"), (h, "h[n]"), (y, "y[n] = x[n] * h[n]")]):
    a.stem(np.arange(len(sig)), sig, basefmt=" ")
    a.axhline(0, color="k", lw=0.8)
    a.set_title(name); a.set_xlabel("n"); a.grid(alpha=0.3, axis="y")
fig.tight_layout()

---
## 4. Multi-panel layouts and shared axes

Time-domain above, frequency-domain below, with a shared x-axis where it makes sense — this
is the standard figure of the course.

In [ ]:
fs = 1000.0
t, N = time_vector(fs, 1.0)
x = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t)
X = np.fft.rfft(x)
f = np.fft.rfftfreq(N, 1 / fs)
mag = 2 * np.abs(X) / N

fig, ax = plt.subplots(2, 1, figsize=(10, 5))
ax[0].plot(t[:150], x[:150])
ax[0].set(xlabel="time [s]", ylabel="amplitude", title="Time domain")

ax[1].plot(f, mag)
ax[1].set(xlabel="frequency [Hz]", ylabel="amplitude", title="Frequency domain", xlim=(0, 300))

for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()

> **`ax.set(...)`** sets several properties in one call. Handy, but note it takes the *short*
> names (`xlabel`, not `set_xlabel`).

### Shared axes

`sharex=True` links the panels: zoom one and the others follow, and the redundant tick labels
are hidden.

In [ ]:
t, N = time_vector(500, 2.0)
signals = {
    "input u(t)": (t >= 0.5).astype(float),
    "output y(t)": 1 - np.exp(-4 * (t - 0.5)) * ((t >= 0.5)),
    "error e(t)": (t >= 0.5).astype(float) - (1 - np.exp(-4 * (t - 0.5)) * (t >= 0.5)),
}

fig, ax = plt.subplots(3, 1, figsize=(9, 5.5), sharex=True)
for a, (name, sig) in zip(ax, signals.items()):
    a.plot(t, np.where(t >= 0.5, sig, 0))
    a.set_ylabel(name, fontsize=9)
    a.grid(alpha=0.3)
ax[-1].set_xlabel("time [s]")        # only the bottom panel needs it
fig.suptitle("First-order system responding to a step at t = 0.5 s")
fig.tight_layout()

### Uneven layouts with `gridspec`

When one panel should be wider or taller than the others, use `subplot_mosaic` — it is far
more readable than raw gridspec indices.

In [ ]:
fig, ax = plt.subplot_mosaic(
    [["wave", "wave"],
     ["spec", "pz"]],
    figsize=(11, 5.5),
)

fs = 500.0
t, N = time_vector(fs, 1.0)
x = np.sin(2 * np.pi * 20 * t) * np.exp(-2 * t)

ax["wave"].plot(t, x); ax["wave"].set_title("waveform"); ax["wave"].set_xlabel("t [s]")

f = np.fft.rfftfreq(N, 1 / fs)
ax["spec"].plot(f, np.abs(np.fft.rfft(x))); ax["spec"].set_xlim(0, 60)
ax["spec"].set_title("spectrum"); ax["spec"].set_xlabel("f [Hz]")

theta = np.linspace(0, 2 * np.pi, 200)
ax["pz"].plot(np.cos(theta), np.sin(theta), "k:", lw=1)
ax["pz"].plot([0.6, 0.6], [0.5, -0.5], "x", ms=9, color="tab:red")
ax["pz"].set_aspect("equal"); ax["pz"].set_title("pole-zero")

for a in ax.values():
    a.grid(alpha=0.3)
fig.tight_layout()

---
## 5. Log scales and decibels

Frequency responses span orders of magnitude in both axes. Linear axes hide everything that
matters.

In [ ]:
f = np.logspace(-1, 3, 500)          # 0.1 Hz to 1 kHz, log-spaced
fc = 10.0
H = 1 / (1 + 1j * f / fc)            # first-order low-pass

fig, ax = plt.subplots(1, 3, figsize=(13, 3))

ax[0].plot(f, np.abs(H))
ax[0].set_title("linear-linear (useless)")

ax[1].semilogx(f, np.abs(H))
ax[1].set_title("semilogx (better)")

ax[2].semilogx(f, 20 * np.log10(np.abs(H)))
ax[2].set_title("semilogx + dB (correct)")
ax[2].set_ylabel("|H| [dB]")
ax[2].axhline(-3, color="r", ls=":", lw=1)
ax[2].axvline(fc, color="r", ls=":", lw=1)
ax[2].annotate("−3 dB at fc", xy=(fc, -3), xytext=(30, 5),
               arrowprops=dict(arrowstyle="->", color="r"), color="r", fontsize=9)

for a in ax:
    a.set_xlabel("frequency [Hz]"); a.grid(alpha=0.3, which="both")
fig.tight_layout()

> **`which='both'` on the grid** draws the minor gridlines too. On a log axis those minor
> lines are what let a reader estimate values between decades — leave them on.

---
## 6. The Bode plot

Magnitude in dB over log frequency, phase in degrees over the same axis, stacked and
x-shared. Write it once as a function and reuse it all semester.

In [ ]:
def bode_plot(f, H, title="", ax=None, label=None, fc=None):
    """Two-panel Bode plot. f in Hz, H complex frequency response.

    Pass an existing `ax` (a length-2 array) to overlay several responses.
    """
    if ax is None:
        fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
    else:
        fig = ax[0].figure

    ax[0].semilogx(f, 20 * np.log10(np.abs(H) + 1e-15), label=label)
    ax[0].set_ylabel("magnitude [dB]")
    ax[0].grid(True, which="both", alpha=0.3)

    ax[1].semilogx(f, np.degrees(np.unwrap(np.angle(H))), label=label)
    ax[1].set_ylabel("phase [deg]")
    ax[1].set_xlabel("frequency [Hz]")
    ax[1].grid(True, which="both", alpha=0.3)
    ax[1].yaxis.set_major_locator(ticker.MultipleLocator(45))

    if fc is not None:
        for a in ax:
            a.axvline(fc, color="r", ls=":", lw=1)
        ax[0].axhline(-3, color="r", ls=":", lw=1)

    if title:
        ax[0].set_title(title)
    if label:
        ax[0].legend(fontsize=8)
    fig.tight_layout()
    return fig, ax

f = np.logspace(-1, 3, 800)
bode_plot(f, 1 / (1 + 1j * f / 10), title="First-order low-pass, fc = 10 Hz", fc=10)

> **`np.unwrap` before plotting phase.** `np.angle` returns values in $(-\pi, \pi]$, so a
> phase that keeps decreasing gets chopped into a sawtooth. `unwrap` removes the $2\pi$
> jumps and gives the continuous curve you actually want to read.

In [ ]:
# Overlaying several orders on one Bode plot -- the roll-off is the point.
fig, ax = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
f = np.logspace(-1, 3, 800)

for order in [1, 2, 4]:
    H = 1 / (1 + 1j * f / 10) ** order
    bode_plot(f, H, ax=ax, label=f"order {order}  ({-20 * order} dB/decade)")

ax[0].set_title("Roll-off is 20 dB/decade per pole")
ax[0].set_ylim(-160, 10)

In [ ]:
# Phase unwrapping, shown side by side so the problem is obvious.
f = np.logspace(-1, 3, 800)
H = 1 / (1 + 1j * f / 10) ** 4

fig, ax = plt.subplots(1, 2, figsize=(11, 3), sharey=False)
ax[0].semilogx(f, np.degrees(np.angle(H)))
ax[0].set_title("np.angle alone -- wraps at ±180°")
ax[1].semilogx(f, np.degrees(np.unwrap(np.angle(H))))
ax[1].set_title("np.unwrap first -- continuous, reaches −360°")
for a in ax:
    a.set_xlabel("frequency [Hz]"); a.set_ylabel("phase [deg]"); a.grid(alpha=0.3, which="both")
fig.tight_layout()

---
## 7. Pole-zero maps

Poles as `×`, zeros as `○`. For continuous-time systems draw the imaginary axis and shade the
stable left half-plane; for discrete-time draw the unit circle. The picture *is* the
stability argument.

In [ ]:
def splane_plot(poles, zeros=(), ax=None, title="s-plane"):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4.6))
    poles, zeros = np.atleast_1d(poles), np.atleast_1d(zeros)

    lim = max(2.0, 1.3 * max(np.abs(np.concatenate([poles, zeros, [1]]))))
    ax.axvspan(-lim, 0, color="tab:green", alpha=0.06)   # stable region
    ax.axhline(0, color="k", lw=0.8)
    ax.axvline(0, color="k", lw=1.2)

    ax.plot(poles.real, poles.imag, "x", ms=11, mew=2, color="tab:red", label="poles")
    if zeros.size:
        ax.plot(zeros.real, zeros.imag, "o", ms=9, mfc="none", mew=2,
                color="tab:blue", label="zeros")

    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xlabel("Re{s}  [1/s]"); ax.set_ylabel("Im{s}  [rad/s]")
    ax.set_title(title); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="upper right")
    return ax

# A damped second-order system: poles at -zeta*wn ± j*wn*sqrt(1-zeta^2)
wn, zeta = 5.0, 0.3
p = np.array([-zeta * wn + 1j * wn * np.sqrt(1 - zeta**2),
              -zeta * wn - 1j * wn * np.sqrt(1 - zeta**2)])
splane_plot(p, title=f"Second-order, ωₙ={wn}, ζ={zeta} (shaded = stable)")

In [ ]:
def zplane_plot(poles, zeros=(), ax=None, title="z-plane"):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4.6))
    poles, zeros = np.atleast_1d(poles), np.atleast_1d(zeros)

    theta = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(theta), np.sin(theta), "k--", lw=1, label="unit circle")
    ax.fill(np.cos(theta), np.sin(theta), color="tab:green", alpha=0.06)
    ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)

    ax.plot(poles.real, poles.imag, "x", ms=11, mew=2, color="tab:red", label="poles")
    if zeros.size:
        ax.plot(zeros.real, zeros.imag, "o", ms=9, mfc="none", mew=2,
                color="tab:blue", label="zeros")

    lim = max(1.4, 1.25 * np.max(np.abs(np.concatenate([poles, zeros, [1]]))))
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}")
    ax.set_title(title); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="upper right")
    return ax

fig, ax = plt.subplots(1, 3, figsize=(14, 4.4))
zplane_plot(np.array([0.7 + 0.5j, 0.7 - 0.5j]), np.array([-1.0]), ax=ax[0],
            title="stable (|p| = 0.86 < 1)")
zplane_plot(np.array([1.05 + 0.3j, 1.05 - 0.3j]), ax=ax[1],
            title="unstable (|p| = 1.09 > 1)")
zplane_plot(np.array([0.0, 0.0]), np.array([1.0, -1.0]), ax=ax[2],
            title="FIR: all poles at the origin")
fig.tight_layout()

> **Read the middle panel.** Poles outside the unit circle mean the impulse response grows
> without bound. In the s-plane the equivalent statement is "poles in the right half-plane".
> Getting comfortable reading these two pictures is most of what stability analysis asks of
> you.

---
## 8. Spectrograms and 2-D data

When the spectrum changes over time you need a third dimension: colour. `pcolormesh` is the
right tool (it handles non-uniform axes and puts values in the correct cells); `imshow` is
faster but assumes a uniform grid.

In [ ]:
from scipy import signal as sig

fs = 4000.0
t, N = time_vector(fs, 4.0)
chirp = sig.chirp(t, f0=50, f1=1500, t1=4.0, method="linear")
chirp += 0.3 * np.sin(2 * np.pi * 800 * t)          # a constant tone on top

f_ax, t_ax, Sxx = sig.spectrogram(chirp, fs, nperseg=256, noverlap=192)
Sxx_db = 10 * np.log10(Sxx + 1e-12)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, chirp, lw=0.3)
ax[0].set(xlabel="time [s]", ylabel="amplitude", title="Chirp + steady 800 Hz tone")

mesh = ax[1].pcolormesh(t_ax, f_ax, Sxx_db, shading="gouraud", cmap="magma",
                        vmin=Sxx_db.max() - 70, vmax=Sxx_db.max())
ax[1].set(xlabel="time [s]", ylabel="frequency [Hz]", title="Spectrogram")
fig.colorbar(mesh, ax=ax[1], label="power [dB]")
fig.tight_layout()

> **Always clip the colour range.** `vmin=max-70` throws away 70 dB of noise floor so the
> structure is visible. Without it, one loud sample sets the scale and everything else is
> black. And always attach a `colorbar` with a labelled unit — a colour map without a scale
> is decoration, not data.

### Colormap choice matters

Use **perceptually uniform** maps (`viridis`, `magma`, `cividis`) for magnitude data, and
**diverging** maps (`RdBu`, `coolwarm`) only when zero is a meaningful midpoint. Avoid
`jet`: it invents visual edges that are not in the data.

In [ ]:
data = Sxx_db[:80, :]
fig, ax = plt.subplots(1, 4, figsize=(14, 2.6))
for a, cmap in zip(ax, ["viridis", "magma", "jet", "RdBu"]):
    a.pcolormesh(data, cmap=cmap, vmin=data.max() - 70, vmax=data.max())
    a.set_title(cmap, fontsize=10); a.set_xticks([]); a.set_yticks([])
fig.suptitle("Same data, four colormaps -- 'jet' fabricates banding that is not there", y=1.06)
fig.tight_layout()

---
## 9. Annotation: making the plot say something

An unannotated plot makes the reader do work you should have done. Mark the cutoff, label
the peak, shade the passband.

In [ ]:
fs = 1000.0
t, N = time_vector(fs, 1.0)
x = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t)
f = np.fft.rfftfreq(N, 1 / fs)
mag = 2 * np.abs(np.fft.rfft(x)) / N

peak_idx = np.argmax(mag)

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(f, mag, lw=1.2)
ax.set_xlim(0, 300)

# Shade a band of interest.
ax.axvspan(40, 60, color="tab:green", alpha=0.12)
ax.text(50, mag.max() * 0.95, "band of interest", ha="center", fontsize=9, color="tab:green")

# Arrow annotation on the peak.
ax.annotate(f"peak: {f[peak_idx]:.0f} Hz, {mag[peak_idx]:.2f} V",
            xy=(f[peak_idx], mag[peak_idx]),
            xytext=(f[peak_idx] + 60, mag[peak_idx] * 0.85),
            arrowprops=dict(arrowstyle="->", lw=1.2),
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7"))

# A reference line with an inline label.
ax.axhline(0.5, color="tab:red", ls="--", lw=1)
ax.text(295, 0.52, "0.5 V", ha="right", color="tab:red", fontsize=9)

ax.set(xlabel="frequency [Hz]", ylabel="amplitude [V]", title="An annotated spectrum")
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# fill_between: showing a tolerance band or a confidence interval.
rng = np.random.default_rng(1)
t, N = time_vector(200, 2.0)
trials = np.array([np.sin(2 * np.pi * 2 * t) + rng.normal(0, 0.25, N) for _ in range(40)])
mean, std = trials.mean(0), trials.std(0)

fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(t, mean - 2 * std, mean + 2 * std, alpha=0.2, label="±2σ")
ax.fill_between(t, mean - std, mean + std, alpha=0.35, label="±1σ")
ax.plot(t, mean, "k", lw=1.5, label="mean of 40 trials")
ax.set(xlabel="time [s]", ylabel="amplitude")
ax.legend(ncol=3, fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()

---
## 10. Styling, rcParams, and colour

Set your defaults once at the top of a notebook rather than repeating arguments on every
call.

In [ ]:
print("A few built-in styles:")
print([s for s in plt.style.available if not s.startswith("_")][:14])

t, N = time_vector(300, 1.0)

# A style must wrap figure *creation*, not just the plotting calls -- most of what a style
# controls (background, grid, spines, fonts) is decided when the Axes is built.
for style in ["default", "ggplot", "bmh"]:
    if style != "default" and style not in plt.style.available:
        continue
    with plt.style.context(style):
        fig, ax = plt.subplots(figsize=(9, 2))
        ax.plot(t, np.sin(2 * np.pi * 3 * t), label="sin")
        ax.plot(t, np.cos(2 * np.pi * 3 * t), label="cos")
        ax.set_title(f"style: {style}", fontsize=10)
        ax.legend(fontsize=8, ncol=2)
        fig.tight_layout()
        plt.show()

In [ ]:
# Persistent defaults for the whole notebook.
plt.rcParams.update({
    "figure.figsize": (9, 3.2),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,      # removing the top/right spines de-clutters
    "axes.spines.right": False,
    "font.size": 10,
    "axes.titlesize": 11,
    "legend.frameon": False,
    "lines.linewidth": 1.4,
})

fig, ax = plt.subplots()
ax.plot(t, np.sin(2 * np.pi * 3 * t), label="sin")
ax.plot(t, np.cos(2 * np.pi * 3 * t), label="cos")
ax.set(xlabel="time [s]", ylabel="amplitude", title="With the new defaults applied")
ax.legend()

In [ ]:
# The default colour cycle, and how to set your own.
cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
print("default cycle:", cycle)

fig, ax = plt.subplots(figsize=(9, 2))
for i, c in enumerate(cycle):
    ax.barh(0, 1, left=i, color=c, height=0.8)
    ax.text(i + 0.5, 0, f"C{i}", ha="center", va="center", color="w", fontsize=9)
ax.set_xlim(0, len(cycle)); ax.set_yticks([]); ax.set_title("tab10 colour cycle")
ax.grid(False)

> **A colour caution.** Roughly 8% of men have some form of red-green colour deficiency. If a
> figure distinguishes traces *only* by red vs. green, some readers cannot read it. Vary the
> line style or add markers as a second channel.

---
## 11. Saving figures for reports

Two rules: **vector for line art**, and **`bbox_inches='tight'`** so labels are not cropped.

In [ ]:
import os
os.makedirs("../scratch", exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 3))
f = np.logspace(-1, 3, 500)
ax.semilogx(f, 20 * np.log10(np.abs(1 / (1 + 1j * f / 10))))
ax.set(xlabel="frequency [Hz]", ylabel="|H| [dB]", title="Saved figure")
ax.grid(alpha=0.3, which="both")

fig.savefig("../scratch/response.png", dpi=200, bbox_inches="tight")   # raster, for slides
fig.savefig("../scratch/response.pdf", bbox_inches="tight")            # vector, for LaTeX
fig.savefig("../scratch/response.svg", bbox_inches="tight")            # vector, for the web

for name in ["response.png", "response.pdf", "response.svg"]:
    size = os.path.getsize(f"../scratch/{name}") / 1024
    print(f"{name:<15} {size:7.1f} KB")

| format | use it for |
|--------|-----------|
| **PDF / SVG** | anything with lines and text — LaTeX reports, lab writeups. Infinite zoom, small files. |
| **PNG @ 200+ dpi** | slides, GitHub READMEs, anything that must be an image. |
| **JPEG** | never, for plots. It smears text and thin lines. |

For a spectrogram or other image-like panel, PNG is correct even in a report — a vector
format would embed millions of tiny rectangles and produce a 50 MB file.

---
## 12. Animation

Animation is genuinely useful for exactly one thing in this course: showing convolution as
the kernel slides. Here it is.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

x = np.zeros(60); x[10:30] = 1.0                 # a rectangular pulse
h = np.exp(-np.arange(20) / 5.0)                 # a decaying exponential kernel
y = np.convolve(x, h)

fig, ax = plt.subplots(2, 1, figsize=(9, 4.5))
ax[0].plot(x, label="x[n]")
flip_line, = ax[0].plot([], [], "r", label="h[k−n] (flipped, sliding)")
ax[0].set_xlim(-20, 80); ax[0].set_ylim(-0.1, 1.5); ax[0].legend(fontsize=8, loc="upper right")

out_line, = ax[1].plot([], [], "tab:green")
ax[1].set_xlim(-20, 80); ax[1].set_ylim(0, y.max() * 1.1)
ax[1].set_xlabel("n"); ax[1].set_ylabel("y[n]")

def frame(n):
    k = np.arange(n - len(h) + 1, n + 1)
    flip_line.set_data(k, h[::-1])
    out_line.set_data(np.arange(n + 1), y[:n + 1])
    return flip_line, out_line

anim = animation.FuncAnimation(fig, frame, frames=len(y), interval=60, blit=True)
plt.close(fig)                      # suppress the static duplicate figure
HTML(anim.to_jshtml())

> **`plt.close(fig)` then `HTML(anim.to_jshtml())`** is the incantation. Without the close
> you get a stray static figure above the animation. `to_jshtml()` embeds a player with
> scrubbing controls; `to_html5_video()` produces a smaller file but needs `ffmpeg` installed.

---
## 13. Exercises

**1. A reusable figure function.** Write `plot_signal_and_spectrum(t, x, fs, title)` that
produces a clean two-panel figure with correct labels, units, and amplitude-scaled spectrum.
Use it for the rest of the course.

**2. Recreate a textbook figure.** Pick any figure from your Signals and Systems textbook and
reproduce it as closely as you can, including annotations. You will learn more from matching
someone else's layout than from inventing your own.

**3. Interactive pole placement.** Use `ipywidgets` (`from ipywidgets import interact`) to
build a slider for $\zeta$ and $\omega_n$ that live-updates a three-panel figure: pole
locations, step response, and Bode magnitude.

**4. Waterfall plot.** Take `data/accelerometer.csv`, compute a spectrum over each 2-second
window, and plot them stacked with a vertical offset per window (a "waterfall"). Compare it
against a `pcolormesh` spectrogram of the same data — when is each more readable?

**5. Colorblind-safe redesign.** Take the four-order Bode overlay from §6 and redesign it so
it remains fully readable in greyscale. Print it to check.

**6. Publication figure.** Produce a single figure of the `ecg_like.csv` signal showing: the
raw trace, the trace after removing baseline wander, and the spectrum of both — with a
shared style, one legend, and saved as a PDF at exactly 8 cm width (the standard single-column
width for IEEE papers).

**7. Annotate the aliasing plot.** Rebuild the aliasing demonstration from notebook 01 and
annotate it so a reader who has never seen aliasing understands it without the caption.

---

### Quick reference

| Task | Call |
|------|------|
| figure + axes | `fig, ax = plt.subplots(rows, cols, figsize=(w, h))` |
| shared x | `plt.subplots(..., sharex=True)` |
| named layout | `plt.subplot_mosaic([["a","a"],["b","c"]])` |
| discrete signal | `ax.stem(n, x, basefmt=" ")` |
| log frequency | `ax.semilogx(f, y)` |
| dB | `20*np.log10(np.abs(H))` for amplitude, `10*np.log10(P)` for power |
| continuous phase | `np.degrees(np.unwrap(np.angle(H)))` |
| 2-D field | `ax.pcolormesh(x, y, Z, shading="gouraud", cmap="magma")` |
| colour scale | `fig.colorbar(mesh, ax=ax, label="dB")` |
| reference line | `ax.axhline(y)`, `ax.axvline(x)` |
| shaded band | `ax.axvspan(x0, x1, alpha=0.1)`, `ax.fill_between(x, lo, hi)` |
| arrow label | `ax.annotate(txt, xy=..., xytext=..., arrowprops=dict(arrowstyle="->"))` |
| fix spacing | `fig.tight_layout()` |
| save | `fig.savefig(path, dpi=200, bbox_inches="tight")` |
| global defaults | `plt.rcParams.update({...})` |